In [ ]:
# =============================================================================
# REGRESSION TRAINING NOTEBOOK
# Predicting HOMO and LUMO Energy Levels for Organic Photovoltaics
# Best Training Pipeline - No Grid Search (Optimal Parameters Hardcoded)
# =============================================================================

import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from rdkit import Chem
from rdkit.Chem import Descriptors

from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import Ridge
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import LocalOutlierFactor

print("✓ All imports successful")

In [ ]:
# =============================================================================
# OPTIMAL PARAMETERS - Found via grid search
# =============================================================================

# HOMO: LOF with k=30 neighbors, 100 samples, 60 features, Ridge alpha=0.5
# CV MAE: 0.0415
HOMO_CONFIG = {
    'method': 'lof',
    'lof_neighbors': 30,
    'n_samples': 100,
    'k_features': 60,
    'ridge_alpha': 0.5
}

# LUMO: GMM with 4 components, 100 samples, 50 features, Ridge alpha=1.5
# CV MAE: 0.0754
LUMO_CONFIG = {
    'method': 'gmm',
    'gmm_components': 4,
    'n_samples': 100,
    'k_features': 50,
    'ridge_alpha': 1.5
}

print("HOMO Configuration:")
for k, v in HOMO_CONFIG.items():
    print(f"  {k}: {v}")

print("\nLUMO Configuration:")
for k, v in LUMO_CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# Load combined dataset (HOPV + SQL)
df_full = pd.read_csv('../../../data/002_processed_data/xy_df_full.csv')
df_full = df_full.dropna(subset=['homo', 'lumo'])

# Extract features and targets
descriptor_cols = [col for col in df_full.columns if col not in ['smiles', 'homo', 'lumo', 'stability']]

X = df_full[descriptor_cols].values
y_homo = df_full['homo'].values
y_lumo = df_full['lumo'].values

print(f"Training dataset: {len(df_full)} samples")
print(f"Features: {len(descriptor_cols)}")
print(f"\nHOMO: mean={y_homo.mean():.3f}, std={y_homo.std():.3f}, range=[{y_homo.min():.3f}, {y_homo.max():.3f}]")
print(f"LUMO: mean={y_lumo.mean():.3f}, std={y_lumo.std():.3f}, range=[{y_lumo.min():.3f}, {y_lumo.max():.3f}]")

In [ ]:
# Load Kaggle test set for distribution matching
df_kaggle = pd.read_csv('../../../data/003_kaggle_competition_data/25261_regression_sample_submission.csv')

# Compute descriptors
descriptor_names = [x[0] for x in Descriptors._descList]

def compute_descriptors(smiles_str):
    mol = Chem.MolFromSmiles(smiles_str)
    if mol is None:
        return [np.nan] * len(descriptor_names)
    return [getattr(Descriptors, name)(mol) for name in descriptor_names]

X_kaggle = pd.DataFrame(
    [compute_descriptors(s) for s in df_kaggle['smiles'].tolist()],
    columns=descriptor_names
)

print(f"Kaggle test set: {len(df_kaggle)} samples")

In [ ]:
# Preprocessing pipeline
var_sel = VarianceThreshold(threshold=0)
imp = SimpleImputer(strategy='median')
scaler = StandardScaler()

# Fit and transform training data
X_clean = var_sel.fit_transform(X)
X_imp = imp.fit_transform(X_clean)
X_scaled = scaler.fit_transform(X_imp)

# Transform Kaggle data
X_kaggle_clean = var_sel.transform(X_kaggle)
X_kaggle_imp = imp.transform(X_kaggle_clean)
X_kaggle_scaled = scaler.transform(X_kaggle_imp)

print(f"Features after preprocessing: {X_imp.shape[1]}")
print(f"Train shape: {X_scaled.shape}")
print(f"Kaggle shape: {X_kaggle_scaled.shape}")

In [ ]:
print("="*60)
print("HOMO MODEL - Sample Selection with LOF")
print("="*60)

# Fit LOF on Kaggle distribution
lof_homo = LocalOutlierFactor(n_neighbors=HOMO_CONFIG['lof_neighbors'], novelty=True)
lof_homo.fit(X_kaggle_scaled)

# Score training samples (higher = more similar to Kaggle)
homo_scores = lof_homo.decision_function(X_scaled)

# Select top n samples most similar to Kaggle
idx_homo = np.argsort(homo_scores)[-HOMO_CONFIG['n_samples']:]
X_homo_sub = X_imp[idx_homo]
y_homo_sub = y_homo[idx_homo]

print(f"LOF neighbors: {HOMO_CONFIG['lof_neighbors']}")
print(f"Selected samples: {len(idx_homo)}")
print(f"Score range: [{homo_scores.min():.3f}, {homo_scores.max():.3f}]")

In [ ]:
print("="*60)
print("LUMO MODEL - Sample Selection with GMM")
print("="*60)

# Fit GMM on Kaggle distribution
gmm_lumo = GaussianMixture(n_components=LUMO_CONFIG['gmm_components'], random_state=42)
gmm_lumo.fit(X_kaggle_scaled)

# Score training samples (higher = more similar to Kaggle)
lumo_scores = gmm_lumo.score_samples(X_scaled)

# Select top n samples most similar to Kaggle
idx_lumo = np.argsort(lumo_scores)[-LUMO_CONFIG['n_samples']:]
X_lumo_sub = X_imp[idx_lumo]
y_lumo_sub = y_lumo[idx_lumo]

print(f"GMM components: {LUMO_CONFIG['gmm_components']}")
print(f"Selected samples: {len(idx_lumo)}")
print(f"Score range: [{lumo_scores.min():.3f}, {lumo_scores.max():.3f}]")

In [ ]:
print("="*60)
print("TRAINING HOMO MODEL")
print("="*60)

# Feature selection
sel_homo = SelectKBest(f_regression, k=HOMO_CONFIG['k_features'])
X_homo_sel = sel_homo.fit_transform(X_homo_sub, y_homo_sub)

# Train Ridge model
model_homo = Ridge(alpha=HOMO_CONFIG['ridge_alpha'])
model_homo.fit(X_homo_sel, y_homo_sub)

# Cross-validation
cv_homo = cross_val_score(model_homo, X_homo_sel, y_homo_sub, cv=5, scoring='neg_mean_absolute_error')
mae_homo = -cv_homo.mean()

print(f"Features selected: {HOMO_CONFIG['k_features']}")
print(f"Ridge alpha: {HOMO_CONFIG['ridge_alpha']}")
print(f"CV MAE: {mae_homo:.4f}")
print(f"CV scores: {[-s for s in cv_homo]}")

In [ ]:
print("="*60)
print("TRAINING LUMO MODEL")
print("="*60)

# Feature selection
sel_lumo = SelectKBest(f_regression, k=LUMO_CONFIG['k_features'])
X_lumo_sel = sel_lumo.fit_transform(X_lumo_sub, y_lumo_sub)

# Train Ridge model
model_lumo = Ridge(alpha=LUMO_CONFIG['ridge_alpha'])
model_lumo.fit(X_lumo_sel, y_lumo_sub)

# Cross-validation
cv_lumo = cross_val_score(model_lumo, X_lumo_sel, y_lumo_sub, cv=5, scoring='neg_mean_absolute_error')
mae_lumo = -cv_lumo.mean()

print(f"Features selected: {LUMO_CONFIG['k_features']}")
print(f"Ridge alpha: {LUMO_CONFIG['ridge_alpha']}")
print(f"CV MAE: {mae_lumo:.4f}")
print(f"CV scores: {[-s for s in cv_lumo]}")

In [ ]:
print("="*60)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*60)

# Get feature names after variance threshold
var_mask = var_sel.get_support()
features_after_var = [descriptor_cols[i] for i in range(len(descriptor_cols)) if var_mask[i]]

# HOMO features
selected_indices_h = sel_homo.get_support(indices=True)
homo_features = []
for idx, coef in zip(selected_indices_h, model_homo.coef_):
    homo_features.append({'feature': features_after_var[idx], 'coef': abs(coef)})
homo_features = sorted(homo_features, key=lambda x: x['coef'], reverse=True)

print("Top 10 HOMO features:")
for i, feat in enumerate(homo_features[:10], 1):
    print(f"  {i:2d}. {feat['feature']:30s} (|coef|={feat['coef']:.4f})")

# LUMO features
selected_indices_l = sel_lumo.get_support(indices=True)
lumo_features = []
for idx, coef in zip(selected_indices_l, model_lumo.coef_):
    lumo_features.append({'feature': features_after_var[idx], 'coef': abs(coef)})
lumo_features = sorted(lumo_features, key=lambda x: x['coef'], reverse=True)

print("\nTop 10 LUMO features:")
for i, feat in enumerate(lumo_features[:10], 1):
    print(f"  {i:2d}. {feat['feature']:30s} (|coef|={feat['coef']:.4f})")

# Overlap
homo_top10 = set([f['feature'] for f in homo_features[:10]])
lumo_top10 = set([f['feature'] for f in lumo_features[:10]])
overlap = homo_top10 & lumo_top10
print(f"\nFeatures in both top 10: {overlap if overlap else 'None'}")

In [ ]:
print("="*60)
print("SAVING MODEL WITH JOBLIB")
print("="*60)

os.makedirs('../../../models', exist_ok=True)

model_package = {
    # Preprocessing
    'var_sel': var_sel,
    'imp': imp,
    'scaler': scaler,
    'descriptor_cols': descriptor_cols,
    
    # HOMO
    'model_homo': model_homo,
    'sel_homo': sel_homo,
    'lof_homo': lof_homo,
    'homo_config': HOMO_CONFIG,
    'mae_homo': mae_homo,
    
    # LUMO
    'model_lumo': model_lumo,
    'sel_lumo': sel_lumo,
    'gmm_lumo': gmm_lumo,
    'lumo_config': LUMO_CONFIG,
    'mae_lumo': mae_lumo,
    
    # Clipping ranges
    'homo_min': y_homo.min(),
    'homo_max': y_homo.max(),
    'lumo_min': y_lumo.min(),
    'lumo_max': y_lumo.max(),
}

joblib.dump(model_package, '../../../models/regression_model.joblib')

print("✓ Model saved to: ../../../models/regression_model.joblib")